# EDA for home credit default risk

#### 1. Загрузка библиотек

In [ ]:
import pandas as pd
import numpy as np
import data_profiling
import matplotlib.pyplot as plt


from pathlib import Path
from data_profiling import ProfileReport


In [ ]:
%matplotlib inline
# %pip uninstall -y ydata-profiling
# %pip install --upgrade --default-timeout=120 --retries 10 fg-data-profiling

#### 2. Загрузка данных:

In [ ]:
path_to_data = "/home/usl/PycharmProjects/home-credit-default-risk/data/raw/home-credit-default-risk/"
# path_to_data = "/content/drive/MyDrive/Home_Credit_data/"

application_train_df = pd.read_csv(path_to_data+"application_train.csv")
application_test_df = pd.read_csv(path_to_data+"application_test.csv")
bureau_df = pd.read_csv(path_to_data+"bureau.csv")
bureau_balance_df = pd.read_csv(path_to_data+"bureau_balance.csv")
credit_card_balance_df = pd.read_csv(path_to_data+"credit_card_balance.csv")
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
installments_payments_df = pd.read_csv(path_to_data+"installments_payments.csv")
POS_CASH_balance_df = pd.read_csv(path_to_data+"POS_CASH_balance.csv")
previous_application_df = pd.read_csv(path_to_data+"previous_application.csv")
sample_submission_df = pd.read_csv(path_to_data+"sample_submission.csv")

#### 3. Размеры таблиц

In [ ]:
size_tables = {
    "application_train": application_train_df,
    "application_test_df": application_test_df,
    "bureau": bureau_df,
    "bureau_balance": bureau_balance_df,
    "credit_card_balance": credit_card_balance_df,
    "home_credit_columns_description": home_credit_columns_description_df,
    "installments_payments": installments_payments_df,
    "POS_CASH_balance": POS_CASH_balance_df,
    "previous_application": previous_application_df,
    "sample_submission": sample_submission_df,
}

for name, df in size_tables.items():
    print(name, df.shape)

#### 4. Проведем анализ данных основной таблицы application_train_df, используя ydata-profiling

In [ ]:
profile = ProfileReport(application_train_df, minimal=True)
profile.to_notebook_iframe()

#### 4. Исследование типов признаков

In [ ]:
application_train_df.info(verbose=True, show_counts = True)

In [ ]:
application_train_df.dtypes.value_counts()
cat_col = application_train_df.select_dtypes(include = "object").columns.tolist()
num_col = application_train_df.select_dtypes(include = ["int64", "float64"]).columns.tolist()

В данных есть технические данные SK_ID_CURR, а также Target. После разделения данных на категориальные и числовые выведем 10 первых категориальных и 10 числовых признаков. Уберем из признаков технические данные.

In [ ]:
num_features = [col for col in num_col if col not in ["TARGET", "SK_ID_CURR"]]

In [ ]:
print("Категориальных признаков:", len(cat_col))
print("Числовых признаков:", len(num_features))
print()
print("Первые категориальные признаки:")
print(cat_col[:10])
print("Первые числовые признаки:")
print(num_col[:10])
print("Первые числовые признаки без технических данных:")
print(num_features[:10])

#### 5. Проверка идентификатора SK_ID_CURR текущей заявки в данных
Проверим уникальность и пересечение SK_ID_CURR в train, test. И подумаем, можно ли его считать обычным числовым признаком.

In [ ]:
application_train_df["SK_ID_CURR"].is_unique

In [ ]:
application_test_df["SK_ID_CURR"].is_unique

In [ ]:
train_ids = set(application_train_df["SK_ID_CURR"])
test_ids = set(application_test_df["SK_ID_CURR"])
common_ids = train_ids.intersection(test_ids)
print("Общих ID между train и test:", len(common_ids))

Значения являются уникальными, пересечений нет. Будем использовать идентификатор SK_ID_CURR для объединения таблиц. Как числовой признак не используется.
Также в application_train нет календарной даты заявки (число, месяц год), поэтому будем использовать деление выборки на train, validation случайным способом (hold-out).

#### 6. Работа с дубликатами
Посмотрим, есть полные дубликаты строк в train и test подвыборках:

In [ ]:
print("Полных дубликатов строк в train", application_train_df.duplicated().sum())
print("Полных дубликатов строк в test", application_test_df.duplicated().sum())

Полных дубликатов строк нет, доп обработка не потребуется.

#### 7. Распределение классов в Target

In [ ]:
target_count = application_train_df["TARGET"].value_counts().sort_index()
target_share = application_train_df["TARGET"].value_counts(normalize = True).sort_index()

print("Количество объектов:")
print(target_count)
print("Доли классов:")
print(target_share)

In [ ]:
target_count.plot(kind = "bar")
plt.title("Распределение классов у TARGET")
plt.xlabel("Target")
plt.ylabel("Количество объектов")
plt.xticks(rotation = 0)
plt.show()


Виден сильный дисбаланс классов: класс 0 в 11 раз больше класса 1. Основной метрикой является ROC-AUC. Также будем смотреть Recall, Precision и Confusion Matrix. При делении application_train будем сохранять долю классов в train, validation выборках через параметр stratify = y.

#### 8. Работа с пропусками


In [ ]:
application_train_df["DAYS_EMPLOYED"].value_counts().head(20)

365243 - непонятно, что значит и что делать с этим?


In [ ]:
missing_train = application_train_df.isna().mean() * 100
missing_table = pd.DataFrame({
    "missing_percent": missing_train,
    "dtype": application_train_df.dtypes.astype(str)
})

missing_table = missing_table.sort_values(
    "missing_percent",
    ascending=False
)
display(missing_table.head(30))

Как видно, самые большие пропуски имеют признаки, связанные с характеристикой жилья. Возможно, отсутствие информации также может характеризовать клиента.

In [ ]:
top_missing = missing_table.head(20).sort_values("missing_percent")

top_missing["missing_percent"].plot(kind = "barh", figsize=(8, 6))
plt.title("Топ 20 признаков с наибольшей долей пропусков")
plt.xlabel("Пропуски, %")
plt.ylabel("Признак")
plt.show()

In [ ]:
feature_cols = [
    col for col in application_test_df.columns
    if col in application_train_df.columns
]

missing_compare = pd.DataFrame({
    "train_missing_percent":
        application_train_df[feature_cols].isna().mean() * 100,
    "test_missing_percent":
        application_test_df[feature_cols].isna().mean() * 100,
})

missing_compare["difference"] = (
    missing_compare["test_missing_percent"]
    - missing_compare["train_missing_percent"]
)

missing_compare["abs_difference"] = missing_compare["difference"].abs()

display(
    missing_compare.sort_values(
        "abs_difference",
        ascending=False
    ).head(20)
)

По структуре пропусков в train, test: выделяется сильно признак EXT_SOURCE_1. МЫ видим  дисбаланс значений в тренировочной и тестовой выборке.

#### 8.1 Связь пропусков с target
Проверим, отличается ли доля TARGET=1 между клиентами, у которых значение признака есть, и клиентами, у которых оно пропущено.

In [ ]:
features_to_check = ["EXT_SOURCE_1", "OWN_CAR_AGE", "COMMONAREA_AVG", "YEARS_BUILD_AVG", "BASEMENTAREA_AVG"]

for col in features_to_check:
    missing_target = application_train_df.groupby(application_train_df[col].isna())["TARGET"].mean()

    print(col)
    print(missing_target)
    print()

Для рассмотренных признаков доля TARGET=1 выше среди клиентов с пропущенными значениями, чем среди клиентов, у которых значение присутствует. Отсутствие значений может содержать полезный сигнал для модели. Поэтому для Random Forest при заполнении числовых пропусков медианой можно дополнительно сохранять индикатор пропуска. Для LightGBM числовые значения NaN можно оставить без явного заполнения.

#### 9. Работа с категориальными признаками
Для категориальных признаков проверяем:
- сколько уникальных категорий;
- есть ли пропуски;
- какая категория встречается чаще всего;
- есть ли редкие значения или выбросы.

In [ ]:
cat_rows = []

for col in cat_col:
    counts = application_train_df[col].value_counts(dropna = False, normalize = True)

    cat_rows.append({
        "feature": col,
        "n_unique": application_train_df[col].nunique(dropna = True),
        "missing_percent": application_train_df[col].isna().mean() * 100,
        "most_common": counts.index[0],
        "most_common_share": counts.iloc[0] * 100,
    })

cat_summary = pd.DataFrame(cat_rows)

display(
    cat_summary.sort_values(
        "n_unique",
        ascending = False
    )
)

ORGANIZATION_TYPE имеет 58 категорий и 0% пропусков, это самый большой категориальный признак.
OCCUPATION_TYPE имеет много пропусков, NaN имеет 18%.
Особенно много пропусков в признаках, связанных с жильём (FONDKAPREMONT_MODE, WALLSMATERIAL_MODE, HOUSETYPE_MODE, EMERGENCYSTATE_MODE), а также в OCCUPATION_TYPE.

In [ ]:
cat_missing_features = ["OCCUPATION_TYPE", "FONDKAPREMONT_MODE", "HOUSETYPE_MODE", "WALLSMATERIAL_MODE", "EMERGENCYSTATE_MODE"]

for col in cat_missing_features:

    result = application_train_df.groupby(application_train_df[col].isna())["TARGET"].mean()

    print(col)
    print(result)
    print()

Для большинства рассмотренных категориальных признаков наличие пропуска связано с изменением доли TARGET=1.
Для признаков, связанных с жильём (FONDKAPREMONT_MODE, HOUSETYPE_MODE, WALLSMATERIAL_MODE, EMERGENCYSTATE_MODE), доля проблемных клиентов выше среди строк с пропущенным значением.
OCCUPATION_TYPE показывает обратную зависимость: среди клиентов с пропущенным значением доля TARGET=1 ниже.

In [ ]:
for col in ["ORGANIZATION_TYPE", "OCCUPATION_TYPE"]:

    print(col)

    counts = application_train_df[col].value_counts(dropna = False)

    print(counts.tail(10))
    print()

В ORGANIZATION_TYPE присутствуют очень редкие категории.

In [ ]:
features = ["ORGANIZATION_TYPE", "OCCUPATION_TYPE"]

for col in features:
    result = (application_train_df.groupby(col, dropna = False)["TARGET"].agg(["count", "mean"]))

    result.columns = ["n_clients", "target_rate"]

    result = result.sort_values("target_rate", ascending = False)

    print(col)
    display(result)

Категориальные признаки ORGANIZATION_TYPE и OCCUPATION_TYPE связаны с TARGET: доля проблемных клиентов различается между категориями.
Например, в OCCUPATION_TYPE доля TARGET=1 составляет 17.2% для Low-skill Laborers и около 4.8% для Accountants.
Необходимо применить smoothed target encoding для Random Forest.
Значение XNA в ORGANIZATION_TYPE встречается часто и имеет собственную долю TARGET=1, поэтому пока сохраняем его как отдельную категорию.

#### 10. Числовые признаки

In [ ]:
application_train_df["CODE_GENDER"].value_counts(dropna = False)

XNA встречается 4 раза из 307 511 строк, это мало.

In [ ]:
application_train_df["NAME_INCOME_TYPE"].value_counts(dropna = False)

Видим 4 большие категории и 4 редких категории.

In [ ]:
(application_train_df.groupby("NAME_INCOME_TYPE")["TARGET"].agg(["count", "mean"]).sort_values("mean", ascending = False))

Доля TARGET = 1 заметно отличается между крупными категориями NAME_INCOME_TYP: для Working она составляет 9,6%, для Commercial associate 7,5%, а для Pensioner 5,4%.

Рассмотрим признаки, связанные с деньгами

In [ ]:
money_features = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"]

for col in money_features:
    values = application_train_df[col].dropna()
    upper = values.quantile(0.99)
    values_for_plot = values[values <= upper]
    values_for_plot.hist(bins=  40)

    plt.title(col + " (до 99-го перцентиля)")
    plt.xlabel(col)
    plt.ylabel("Количество клиентов")
    plt.show()

#### 11.Корреляция числовых признаков с Target

In [ ]:
numeric_data = application_train_df.select_dtypes(include = ["int64", "float64"])
target_corr = (numeric_data.corr()["TARGET"].drop(["TARGET", "SK_ID_CURR"]))
top_corr_features = (target_corr.abs().sort_values(ascending = False).head(20).index)
top_corr = target_corr.loc[top_corr_features].sort_values()
display(top_corr)

In [ ]:
top_corr.plot(kind = "barh", figsize = (8, 7))

plt.title("Корреляции числовых признаков с TARGET")
plt.xlabel("Корреляция")
plt.ylabel("Признак")
plt.show()

#### 12. Константные значения

In [ ]:
n_unique = application_train_df.nunique(dropna=False)

constant_features = n_unique[n_unique <= 1]

print("Количество константных признаков:",
      len(constant_features))

display(constant_features)

### Анализ других таблиц
Посмотрим SK_ID_CURR

In [ ]:
bureau_df["SK_ID_CURR"].is_unique

In [ ]:
bureau_df["SK_ID_BUREAU"].is_unique

In [ ]:
bureau_df["SK_ID_CURR"].value_counts().describe()

In [ ]:
previous_application_df["SK_ID_CURR"].value_counts().describe()

In [ ]:
bureau_balance_df["SK_ID_BUREAU"].value_counts().describe()

In [ ]:
train_ids = set(application_train_df["SK_ID_CURR"])
test_ids = set(application_test_df["SK_ID_CURR"])
common_ids = train_ids.intersection(test_ids)
print("Количество общих SK_ID_CURR:", len(common_ids))

In [ ]:
application_train_df["SK_ID_CURR"].is_unique

In [ ]:
application_test_df["SK_ID_CURR"].is_unique

In [ ]:
application_train_df["SK_ID_CURR"].describe()

In [ ]:
application_test_df["SK_ID_CURR"].describe()

Таким образом, в каждой таблице одна строка соответствует одной текущей заявке. Один текущий SK_ID_CURR встречается только 1 раз. То есть train и test занимают практически один и тот же диапазон SK_ID_CURR.

In [ ]:
train_id_df = application_train_df[["SK_ID_CURR"]].copy()
train_id_df["dataset"] = "train"

test_id_df = application_test_df[["SK_ID_CURR"]].copy()
test_id_df["dataset"] = "test"

id_df = pd.concat([train_id_df, test_id_df], ignore_index = True)
id_df["id_bin"] = pd.qcut(id_df["SK_ID_CURR"], q = 10, duplicates="drop")
id_distribution = pd.crosstab(id_df["id_bin"], id_df["dataset"], normalize = "index")

id_distribution

Вывод: SK_ID_CURR уникален внутри train и внутри test, при этом train и test не пересекаются, диапазоны SK_ID_CURR почти одинаковы. test распределен по всему диапазону SK_ID_CURR. Также SK_ID_CURR считаем техническим идентификатором, не будем проводить обучение моделей в качестве обычного числового признака.

Посмотрим другие таблицы, объединим application_train_df с bureau_df.

In [ ]:
home_credit_columns_description_df.columns
home_credit_columns_description_df.head()

In [ ]:
bureau_df.info(verbose=True, show_counts = True)

In [ ]:
bureau_df = pd.read_csv(path_to_data+"bureau.csv")
bureau_df

In [ ]:
application_bureau= application_train_df.merge(bureau_df, on = "SK_ID_CURR", how = "left", indicator = True)
application_bureau

In [ ]:
application_bureau._merge.value_counts()